# Imports and client init

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from dotenv import load_dotenv
import uuid

from linalgo.hub.client import LinalgoClient
from linalgo.annotate.models import Corpus, Entity
from wsd.load_corpus import load_corpus
from linalgo.annotate.serializers import DocumentSerializer

In [ ]:
load_dotenv()
token = os.getenv('LINHUB_TOKEN')
url = "https://linhub.api.linalgo.com/v1"
client = LinalgoClient(token, url)
jack_org_id = "acf7a1aa-ec18-4fa2-a981-a756bc6e6af2"
test_task_id = "635b8e9d-b590-4222-83a0-b46762a9fa58"
test_id = "6667052e-b464-47a9-beca-dd8df8f8c632"
jack_org = client.get_organization(jack_org_id)

In [ ]:
task = client.get_task(test_task_id)

In [ ]:
task.entities

# Load corpus March (superseeding from 3/3/2025)

In [ ]:
new_corpus = load_corpus("fr", test_task_id , jack_org_id, token)
new_corpus

In [ ]:
docs = new_corpus.documents
len(docs)

In [ ]:
docs[0].__dict__

In [ ]:
def get_max_entities(corpus):
    max_entities = 0
    for doc in new_corpus.documents:
        annos = list(doc.annotations)
        entities = set([anno.entity for anno in annos])
        if len(entities) > max_entities:
            max_entities = len(entities)
            print (f"New max entities: {max_entities}")
            print(f"Word: {annos[0].body.text}")
            print(f"Number of annos:  {len(annos)}")
    return max_entities

In [ ]:
get_max_entities(new_corpus)

# Creating task

In [ ]:
#Generate a UID for the task
new_task_id = str(uuid.uuid4())
lang = "fr"
new_corpus = Corpus(
    name=f'Semcor_{lang}',
    description=f'The Semcor wsd corpus for {lang}',
    organization=jack_org)
client.create_corpus(new_corpus, jack_org)
new_task_id, new_corpus.id

In [ ]:
exsting_task_id = "d3ce7764-eb85-4999-b965-c028f539ee33"
entities = client.get_task(exsting_task_id).entities

In [ ]:
def create_and_post_task(
    name: str,
    organization: str,
    task_id : str = str(uuid.uuid4()),
    description: str = None,
    corpus_id: str = None,
    entities: list[Entity] = [],
    ) -> None:


    serialized_entities = [entity.id for entity in entities]

    post_url = url + f"/tasks/"
    data  = {
        "id": task_id,
        "name": name,
        "slug": name,
        "organization": organization,
        "description": description,
        "entities": serialized_entities,
        "corpora": [corpus_id],
    }
    client.post(url = post_url, data= data)
    pass

In [ ]:
create_and_post_task(
    name = "test_task_2",
    organization = jack_org_id,
    task_id = new_task_id,
    corpus_id = new_corpus.id,
    entities = entities
)

In [ ]:
task = client.get_task(new_task_id, verbose=True)

In [ ]:
task.corpora[0]

In [ ]:
task.documents[0].annotations

In [ ]:
updated_corpus = load_corpus("fr", new_task_id , jack_org_id, token)

In [ ]:
for doc in updated_corpus.documents:
    client.create_annotations(list(doc.annotations))

In [ ]:
task = client.get_task(new_task_id, verbose=True)

In [ ]:
serializer = DocumentSerializer(updated_corpus.documents[0])
serializer.serialize()

# adding annotations

In [ ]:
annos = []

for doc in docs:
    for anno in doc.annotations:
        annos.append(anno)
len(annos)

In [ ]:
annos[0].__dict__

In [ ]:
for anno in annos:
    client.create_annotations(anno)